# UPCT Medical Image Segmentation Challenge 2026-27
## Práctica 7

**Asignatura:** Procesado de Imágenes Médicas (521104007)

**Profesor:** Juan Zapata

> Contenido **nuevo** de esta práctica. Copia las celdas de aquí abajo y pégalas **al final** de tu propio notebook (el que empezaste en la Práctica 6) — no repitas las prácticas anteriores, ya las tienes hechas ahí.

## Guía de Sesiones (2 horas por sesión)
| Práctica | Fechas (Grupo A / B) | Objetivo de la Sesión | Checkpoint Visual |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset y formato RLE | 6 imágenes con máscaras + RLE OK |
| ▶ **P7** | 9-11 Nov | Baseline U-Net y 1ª Submission | Gráficas de Loss + Submission Kaggle |
| **P8** | 16-18 Nov | Data Augmentation y mejora | Comparativa Baseline vs Augmented |
| **P9** | 23-25 Nov | Inferencia, Threshold y Errores | 5 imágenes normales + 2 casos de error |
| **P10** | 30 Nov-2 Dic | TTA, Submission Final y Defensa | Mejor Dice Score + Defensa Oral |

> **Regla de Oro:** Según el Art. 7.5 del Reglamento de Evaluación UPCT, la asistencia y validación del Checkpoint en el aula es obligatoria para superar la práctica.


# Práctica 7: Baseline Model (U-Net) y 1ª Submission
## Sesión única (9 Nov Grupo A / 11 Nov Grupo B)

### Objetivos de la sesión:
1. Crear un `Dataset` y `DataLoader` de PyTorch para cargar las imágenes y máscaras.
2. Definir la arquitectura **U-Net** y la función de pérdida (**Dice Loss + BCE**).
3. Entrenar el modelo durante unas pocas épocas (baseline).
4. Realizar la inferencia sobre el conjunto de test y generar el archivo `submission.csv`.

> **CHECKPOINT P7:** Mostrar al profesor las gráficas de pérdida (Loss) decreciendo y la captura de tu primera submission en el Leaderboard de Kaggle.

## Bloque 7.1: Dataset y DataLoader en PyTorch
### ¿Por qué no basta con una lista de arrays?

Hasta ahora habéis trabajado con imágenes cargadas "a mano" (un `cv2.imread` suelto, un bucle `for`). Para entrenar una red neuronal necesitáis algo más:

| Sin `Dataset`/`DataLoader` | Con `Dataset`/`DataLoader` |
|---|---|
| Bucle manual cargando todo en RAM | Carga **bajo demanda**, una muestra cada vez |
| Batches construidos a mano | Agrupa en **batches** automáticamente |
| Barajado manual (o ninguno) | `shuffle=True` reordena en cada época |
| Un solo proceso leyendo disco | `num_workers` carga en **paralelo** |

Con 546 imágenes esto no parece crítico, pero es el mismo patrón que usaréis con datasets de millones de imágenes: **PyTorch no cambia de API**, solo cambia el tamaño del dataset.

### La abstracción: dos piezas con responsabilidades distintas

```
DataFrame (índice)         Dataset                    DataLoader
class, image_path,   →    __getitem__(idx)    →      junta N muestras
mask_path, filename        devuelve 1 muestra          en un batch
                            (imagen, máscara)           (B, C, H, W)
```

| | `Dataset` | `DataLoader` |
|---|---|---|
| **Responde a** | "¿Cómo obtengo la muestra `i`?" | "¿Cómo agrupo y sirvo las muestras?" |
| **Métodos clave** | `__len__`, `__getitem__` | — (es un envoltorio) |
| **Tú decides** | Cómo cargar, preprocesar y devolver 1 muestra | `batch_size`, `shuffle`, `num_workers` |

> **Idea clave:** el `Dataset` no sabe nada de batches. Solo sabe devolver **una** muestra dado un índice. Toda la lógica de agrupar, barajar y paralelizar es responsabilidad del `DataLoader`.

### El reto de la segmentación: pares (imagen, máscara)

En clasificación, `__getitem__` devuelve `(imagen, etiqueta)` — la etiqueta es un número, no necesita transformarse.

En **segmentación** devolvéis `(imagen, máscara)`, y aquí hay una trampa:

- Las transformaciones **geométricas** (resize, flip, rotación) deben aplicarse **igual** a la imagen y a la máscara — si giras la imagen 15° y no giras la máscara, dejan de estar alineadas.
- Las transformaciones **fotométricas** (normalización, brillo, contraste, ruido) solo tienen sentido en la **imagen** — la máscara debe seguir siendo binaria (0/1), nunca "iluminarla" ni "añadirle ruido".

> **Pregunta para pensar:** si en la Práctica 8 rotáis la imagen 15° dentro de `__getitem__` pero se os olvida rotar la máscara, ¿qué Dice Score esperaríais obtener? ¿Alto, bajo, o dependería de la imagen?

### Normalización: la trampa oculta de los encoders preentrenados

Una normalización habitual es dividir por 255 para llevar los píxeles a `[0, 1]`. Es correcta... **pero incompleta** para nuestro caso.

Nuestro modelo (Práctica 7) usa un encoder **ResNet34 preentrenado en ImageNet** (`encoder_weights="imagenet"`). Ese encoder fue entrenado con imágenes normalizadas con la **media y desviación típica de ImageNet**, no con `[0, 1]` a secas:

| Canal | Media | Desviación típica |
|-------|-------|--------------------|
| R | 0.485 | 0.229 |
| G | 0.456 | 0.224 |
| B | 0.406 | 0.225 |

```python
image = image.astype(np.float32) / 255.0
image = (image - mean) / std   # mean, std por canal, arriba
```

Si alimentáis al encoder preentrenado con datos en `[0, 1]` sin centrar, la red "ve" una distribución de entrada distinta a la que aprendió — el entrenamiento seguirá funcionando, pero partiréis de una posición peor y convergeréis más lento (y con Data Augmentation, Albumentations os hace esta normalización automáticamente vía `Normalize()`, así que a partir de la Práctica 8 no la escribiréis vosotros).

### Convención de ejes: `(H, W, C)` vs `(C, H, W)`

- `cv2` / `numpy` representan una imagen como `(Alto, Ancho, Canales)`.
- PyTorch espera `(Canales, Alto, Ancho)` para todo lo que entra a una capa `Conv2d`.

```python
image_tensor = torch.tensor(image).permute(2, 0, 1)   # (H,W,C) → (C,H,W)
```

La máscara es un caso especial: se carga como `(H, W)` (un solo canal, sin dimensión de canal explícita). Antes de meterla en el modelo necesita una dimensión de canal: `(1, H, W)` — por eso veréis `.unsqueeze(0)`.

### Train/Val Split: ¿aleatorio es siempre suficiente?

`train_test_split(df, test_size=0.2)` reparte filas al azar. Pero recordad el "reto oculto" de la Práctica 6: el dataset tiene solo 133 imágenes `normal` de 780. Un split puramente aleatorio no garantiza que esa proporción se mantenga en el conjunto de validación.

> **Pregunta para pensar:** ¿qué pasaría con vuestro `best_threshold` (Práctica 9) si por mala suerte el split dejó casi ninguna imagen `normal` en validación?

*(Pista para quien quiera ir más allá: `train_test_split(..., stratify=df['class'])` mantiene las proporciones de clase en train y val.)*

### Los parámetros del `DataLoader`

| Parámetro | Qué hace | Regla práctica |
|---|---|---|
| `batch_size` | Nº de muestras por paso | Más alto = más rápido, más memoria GPU |
| `shuffle` | Reordena las muestras cada época | `True` en train, `False` en val |
| `num_workers` | Carga en paralelo desde disco | En Colab, 2 workers suele bastar |

> `shuffle=False` en val no es un descuido: así comparáis siempre las mismas muestras en el mismo orden entre épocas.

### Resumen rápido

| Concepto | Idea principal |
|----------|----------------|
| `Dataset` | Devuelve 1 muestra dado un índice (`__getitem__`) |
| `DataLoader` | Agrupa en batches, baraja y paraleliza la carga |
| Imagen vs Máscara | Misma transf. geométrica; normalización solo en la imagen |
| Normalización | `/255` no basta con encoders preentrenados (hace falta ImageNet mean/std) |
| `(H,W,C)` → `(C,H,W)` | Convención numpy/cv2 vs PyTorch — usad `permute` |
| Split train/val | Aleatorio simple puede desequilibrar clases minoritarias (`normal`) |

### Referencias

1. **PyTorch Docs.** *`torch.utils.data.Dataset` and `DataLoader`.* pytorch.org/docs/stable/data.html
2. **Deng, J., et al. (2009).** *ImageNet: A Large-Scale Hierarchical Image Database.* CVPR. *(origen de las estadísticas de normalización mean/std)*
3. **He, K., et al. (2016).** *Deep Residual Learning for Image Recognition.* CVPR. *(ResNet, el encoder que usaréis en la U-Net)*

> Siguiente paso: ahora que sabéis qué debe hacer cada pieza, implementad `BUSIDataset` y sus `DataLoader` en la Tarea 7.1.

## Tarea 7.1: Dataset y DataLoader
Para entrenar en PyTorch, necesitamos empaquetar nuestros datos.
1. Crea una clase `BUSIDataset` que herede de `torch.utils.data.Dataset`.
2. En `__getitem__`, carga la imagen (RGB) y la máscara (escala de grises).
3. Redimensiona ambas a `IMG_SIZE` (ej. 256x256).
4. Normaliza la imagen (dividir por 255.0) y binariza la máscara (umbral > 127).
5. Devuelve la imagen como tensor `(C, H, W)` y la máscara como tensor `(1, H, W)`.
6. Crea los `DataLoader` para train y validation (usa un 20% para val).

> **Pista:** Usa `torch.tensor(img).permute(2, 0, 1)` para cambiar el formato de imagen.

In [ ]:
# ============================================================
# TAREA 7.1: DATASET Y DATALOADER
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np

# Hiperparámetros
IMG_SIZE = 256
BATCH_SIZE = 16

# ESCRIBE TU CÓDIGO AQUÍ
class BUSIDataset(Dataset):
    def __init__(self, dataframe, img_size=IMG_SIZE):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        # Tu código aquí
        pass

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Cargar imagen y máscara
        # Tu código aquí

        # 2. Redimensionar a IMG_SIZE
        # Tu código aquí

        # 3. Normalizar imagen y binarizar máscara
        # Tu código aquí

        # 4. Convertir a tensores de PyTorch
        # Tu código aquí

        return image_tensor, mask_tensor

# Dividir el dataset (80% train, 20% val)
# Tu código aquí para crear train_df y val_df a partir de df_train

# Crear Datasets y DataLoaders
# Tu código aquí

## Bloque 7.2: Arquitectura U-Net y Función de Pérdida
### Clasificación vs Detección vs Segmentación

| Tarea | Pregunta que responde | Salida |
|-------|----------------------|--------|
| **Clasificación** | ¿Hay tumor? | Etiqueta: "benigno" / "maligno" |
| **Detección** | ¿Dónde está el tumor? | Bounding box |
| **Segmentación** | ¿Qué píxeles son tumor? | Máscara píxel a píxel |

### ¿Por qué segmentación y no clasificación?
- En medicina, el **tamaño y forma** del tumor importan (estadiaje)
- Permite calcular **volumen**, **bordes irregulares**, **invasión**
- Es la base de la **radiómica** y el diagnóstico asistido

### El reto específico: Ultrasonido mamario (BUS)
- Imágenes **ruidosas** y de **bajo contraste**
- Tumores **pequeños** (a veces < 5% de la imagen)
- Artefactos: sombras acústicas, reverberaciones
- **Clase "normal"**: el modelo debe aprender a NO detectar nada

### ¿Qué es la Segmentación Semántica?

Antes de hablar de U-Net, recordemos qué estamos haciendo:

| Tarea | Entrada | Salida |
|-------|---------|--------|
| **Clasificación** | Imagen | Una etiqueta ("benigno", "maligno") |
| **Detección** | Imagen | Bounding boxes + etiquetas |
| **Segmentación** | Imagen | **Máscara píxel a píxel** |

En nuestro caso, la salida es una **máscara binaria** del mismo tamaño que la imagen de entrada, donde cada píxel tiene valor:
- `1` → pertenece al tumor
- `0` → es tejido sano (fondo)

### Arquitectura U-Net

#### Origen

Propuesta por **Ronneberger et al. (2015)** en el paper *"U-Net: Convolutional Networks for Biomedical Image Segmentation"*. Fue diseñada específicamente para **segmentación biomédica** con pocos datos de entrenamiento.

#### Idea Clave: Forma de "U"

La arquitectura tiene forma de **U** porque combina dos caminos:

```
Entrada → [ENCODER] → [BOTTLENECK] → [DECODER] → Salida
   ↓          ↓              ↓             ↓
  572x572   28x28          4x4           572x572
```

### Encoder (Camino Contractivo)

El encoder es similar a una CNN clásica de clasificación. Su trabajo es **extraer características** y reducir la resolución espacial:

```
Input (256x256x3)
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 128x128
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 64x64
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 32x32
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 16x16
    │
    ▼
[Conv 3x3] → [Conv 3x3]                  → 8x8  (Bottleneck)
```

**¿Qué hace el encoder?**
- Extrae características de **bajo nivel** (bordes, texturas) en las primeras capas
- Extrae características de **alto nivel** (formas, patrones complejos) en las capas profundas
- **Reduce** la resolución espacial pero **aumenta** los canales (más información semántica)

### Bottleneck (Cuello de Botella)

Es el punto más profundo de la red. Aquí la imagen se ha reducido mucho espacialmente, pero contiene la **información semántica más abstracta**.

### Decoder (Camino Expansivo)

El decoder **reconstruye** la resolución espacial píxel a píxel:

```
Bottleneck (8x8)
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 16x16
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 32x32
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 64x64
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 128x128
    │
    ▼
[Conv 1x1] + Sigmoid                         → 256x256x1 (Máscara)
```

### Skip Connections: El "Secreto" de U-Net

Aquí está la **innovación clave**: las **conexiones saltarinas** (skip connections) que unen el encoder con el decoder:

```
Encoder                    Decoder
   │                          │
   ├──[feat1]──────────────►[+ merge]──►
   │                          │
   ├──[feat2]──────────────►[+ merge]──►
   │                          │
   ├──[feat3]──────────────►[+ merge]──►
   │                          │
   └──[feat4]──────────────►[+ merge]──►
                              │
                           [Output]
```

**¿Por qué son tan importantes?**

| Sin Skip Connections | Con Skip Connections |
|---------------------|---------------------|
| Solo información semántica | Información semántica **+** localización precisa |
| Bordes borrosos | Bordes nítidos |
| Pierde detalles finos | Recupera detalles finos |

> **Analogía:** Imagina que el decoder es un pintor que tiene que reconstruir un cuadro. El encoder le da la "idea general" (qué hay pintado), pero las skip connections le pasan los "bocetos originales" para que pueda pintar con precisión.

### Función de Pérdida: Dice Loss + BCE

En nuestro notebook usamos una **función de pérdida combinada**:

```python
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum()
        dice_loss = 1 - (2. * intersection + self.smooth) / \
                    (probs.sum() + targets.sum() + self.smooth)

        return bce_loss + dice_loss
```

### 1. Binary Cross Entropy (BCE)

Es la pérdida clásica para clasificación binaria, aplicada **píxel a píxel**:

$$\text{BCE} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]$$

Donde:
- $y_i$ = valor real del píxel $i$ (0 o 1)
- $\hat{y}_i$ = predicción del modelo para el píxel $i$
- $N$ = número total de píxeles

**Intuición:**
- Si el píxel real es `1` y predices `0.9` → penalización **baja**
- Si el píxel real es `1` y predices `0.1` → penalización **alta**

**Problema del BCE en segmentación:**
- Las imágenes médicas tienen **desequilibrio de clases** extremo
- Ejemplo: en una imagen de 256×256 = 65,536 píxeles, quizás solo 500 son tumor (~0.7%)
- El BCE trata todos los píxeles por igual → el modelo puede aprender a predecir todo como "fondo" y tener buen accuracy

### 2. Dice Loss

Basada en el **Coeficiente de Dice** (o *F1 Score*):

$$\text{Dice} = \frac{2 \cdot |A \cap B|}{|A| + |B|}$$

Donde:
- $A$ = píxeles predichos como tumor
- $B$ = píxeles reales de tumor
- $|A \cap B|$ = intersección (píxeles correctamente clasificados)

La **Dice Loss** es simplemente:

$$\text{Dice Loss} = 1 - \text{Dice} = 1 - \frac{2 \cdot |A \cap B| + \epsilon}{|A| + |B| + \epsilon}$$

El término $\epsilon$ (smooth) evita **división por cero** cuando ambas máscaras están vacías.

**Intuición visual:**

```
    A (Predicción)          B (Ground Truth)        A ∩ B (Intersección)
    ┌─────────┐             ┌─────────┐             ┌─────────┐
    │  ████   │             │   ████  │             │   ███   │
    │  ████   │             │   ████  │             │   ███   │
    │  ████   │             │   ████  │             │   ███   │
    └─────────┘             └─────────┘             └─────────┘

    Dice = 2·(intersección) / (|A| + |B|)
```

**Ventajas del Dice Loss:**
- **Robusto al desbalance** de clases
- Optimiza directamente la **métrica que nos importa** (Dice Score)
- Funciona bien con **objetos pequeños** (tumores pequeños)

**Desventajas del Dice Loss:**
- Inestable al inicio del entrenamiento (gradientes ruidosos)
- No considera la **localización espacial** de cada píxel individual

### ¿Por qué combinarlas?

| Pérdida | Fortaleza | Debilidad |
|---------|-----------|-----------|
| **BCE** | Gradientes estables, considera cada píxel | Sensible al desbalance |
| **Dice** | Robusta al desbalance | Gradientes inestables al inicio |

**Combinándolas obtenemos lo mejor de ambos mundos:**

$$\mathcal{L}_{total} = \mathcal{L}_{BCE} + \mathcal{L}_{Dice}$$

- El **BCE** proporciona gradientes estables desde el primer batch
- El **Dice** empuja al modelo a maximizar la métrica de segmentación
- Juntas convergen más rápido y a mejores resultados

## Aplicación a Nuestro Challenge

En el *UPCT Medical Image Segmentation Challenge*:

```python
# Modelo U-Net con encoder ResNet34 pre-entrenado
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

# Pérdida combinada
criterion = DiceBCELoss(smooth=1.0)

# Optimizador Adam
optimizer = optim.Adam(model.parameters(), lr=1e-4)
```

### Flujo completo:

```
Imagen (256×256×3)
    │
    ▼
[Encoder ResNet34] → Extrae features jerárquicas
    │
    ▼
[Decoder] → Reconstruye máscara con skip connections
    │
    ▼
Logits (256×256×1)
    │
    ▼
[Sigmoid] → Probabilidades [0, 1]
    │
    ▼
[Umbral 0.5] → Máscara binaria
    │
    ▼
[DiceBCELoss] → Comparar con ground truth
    │
    ▼
[Backpropagation] → Actualizar pesos
```


### Referencias

1. **Ronneberger, O., Fischer, P., & Brox, T. (2015).** *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
2. **Sudre, C. H., et al. (2017).** *Generalised Dice overlap as a deep learning loss function for highly unbalanced segmentations.* Deep Learning in Medical Image Analysis.
3. **Milletari, F., Navab, N., & Ahmadian, S. A. (2016).** *V-Net: Fully convolutional neural networks for volumetric medical image segmentation.* 3DV.


### Resumen Rápido

| Concepto | Idea Principal |
|----------|----------------|
| **U-Net** | Encoder + Decoder + Skip Connections |
| **Encoder** | Extrae características, reduce resolución |
| **Decoder** | Reconstruye la máscara píxel a píxel |
| **Skip Connections** | Preservan información espacial de alto nivel |
| **BCE** | Pérdida píxel a píxel, estable pero sensible al desbalance |
| **Dice Loss** | Robusta al desbalance, optimiza la métrica final |
| **Dice + BCE** | Lo mejor de ambos mundos |

> **Siguiente paso:** En la práctica, implementaremos U-Net con `segmentation_models_pytorch` y entrenaremos con la pérdida combinada. ¡A codear!


## Tarea 7.2: Modelo U-Net y Función de Pérdida
Vamos a usar la librería `segmentation_models_pytorch` (smp) para crear una U-Net con un backbone pre-entrenado (ResNet34).
1. Define el modelo `smp.Unet` con `encoder_weights='imagenet'` y `activation=None` (importante para la pérdida ya que usamos BCEWithLogitsLoss).
2. Define una clase de pérdida personalizada `DiceBCELoss` que combine `BinaryCrossEntropyWithLogitsLoss` y `DiceLoss`.
3. Configura el optimizador `Adam`.

In [ ]:
# ============================================================
# TAREA 7.2: MODELO Y LOSS
# ============================================================
# Instalar las librerías necesarias para la segmentación
!pip install -q segmentation_models_pytorch timm

import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {DEVICE}")

# 1. Definir el modelo U-Net
# Tu código aquí (usa smp.Unet)
model = ...
model = model.to(DEVICE)

# 2. Definir la función de pérdida combinada (Dice + BCE)
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # Tu código aquí: calcular BCE y Dice Loss y sumarlos
        pass

criterion = DiceBCELoss().to(DEVICE)

# 3. Optimizador
# Tu código aquí (usa optim.Adam con lr=1e-4)
optimizer = ...

## Bloque 7.3: El Bucle de Entrenamiento en PyTorch
### Anatomía de un paso de entrenamiento

  Cada batch pasa por las mismas cinco líneas, siempre en el mismo orden. Conviene memorizar el orden porque los tres bugs más comunes de PyTorch vienen de romperlo:

  ```python
  optimizer.zero_grad()   # 1. Poner a cero los gradientes del paso anterior
  logits = model(images)  # 2. Forward pass
  loss = criterion(logits, masks)  # 3. Calcular la pérdida
  loss.backward()         # 4. Backward pass (calcula gradientes)
  optimizer.step()        # 5. Actualizar los pesos con esos gradientes
  ```

  | Paso | Qué hace | Qué pasa si lo olvidas u omites |
  |------|----------|----------------------------------|
  | `optimizer.zero_grad()` | Borra los gradientes acumulados del batch anterior | Los gradientes se **suman** entre batches; el modelo aprende mal y de forma errática |
  | `model(images)` | Calcula la predicción (logits, sin sigmoide todavía) | — |
  | `criterion(logits, masks)` | Compara predicción vs. máscara real | — |
  | `loss.backward()` | Calcula cuánto contribuye cada peso al error (gradientes) | Sin esto, `optimizer.step()` no tiene nada que actualizar |
  | `optimizer.step()` | Mueve los pesos en la dirección que reduce el error | Sin esto, el modelo nunca cambia aunque calcules gradientes |


### Épocas, batches y pasos

Tres palabras que se confunden fácilmente:

| Término | Definición | En nuestro caso |
|---------|------------|------------------|
| **Paso (step)** | Una actualización de pesos = un batch procesado | `optimizer.step()` una vez |
| **Batch** | Un grupo de `batch_size` muestras | 16 imágenes (`BATCH_SIZE = 16`) |
| **Época (epoch)** | Una pasada completa por todo el `train_loader` | `len(train_loader)` pasos |

`len(train_loader)` no es el número de imágenes, es el número de **batches**: aproximadamente `num_muestras / batch_size`. Si tenéis 437 muestras de train y `batch_size=16`, cada época son 28 pasos, no 437.

### La diferencia de `model.train()` frente a `model.eval()`

Vuestro encoder (ResNet34) usa capas de **BatchNorm**. Estas capas se comportan distinto según el modo:

| Modo | BatchNorm usa... | Cuándo activarlo |
|------|-------------------|--------------------|
| `model.train()` | Estadísticas del batch actual (media/varianza) | Durante el entrenamiento |
| `model.eval()` | Estadísticas acumuladas de todo el entrenamiento | Durante validación e inferencia |

Si evaluáis con el modelo en modo `train()`, el resultado depende de qué otras imágenes haya en ese batch de validación — dos ejecuciones con el mismo modelo pueden dar Dice Scores distintos. Por eso, antes de cada fase de validación es obligatorio llamar a `model.eval()`, y antes de volver a entrenar, `model.train()` otra vez.

### Como se comporta `torch.no_grad()` durante la validación

Al entrenar, PyTorch construye un grafo de cómputo para poder calcular gradientes con `backward()`. Ese grafo consume memoria y tiempo.

En validación **no vais a llamar a `backward()`**, así que ese grafo es trabajo desperdiciado. `torch.no_grad()` le dice a PyTorch que no lo construya:

```python
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        ...
```

Sin `torch.no_grad()`, el bucle de validación sigue siendo *correcto* (el resultado numérico no cambia), pero es más lento y puede agotar la memoria de la GPU en datasets grandes.


### Promediar la métrica por época

Dentro del bucle vais acumulando la pérdida y el Dice **por batch**:

```python
epoch_train_loss += loss.item()
```

Al terminar todos los batches de la época, hay que dividir por el número de batches para obtener el promedio real:

```python
epoch_train_loss /= len(train_loader)
```

Si os olvidáis de esta división, el número que imprimís no es una pérdida por batch sino una **suma acumulada**, que crece con cada batch dentro de la misma época y no es comparable entre épocas con distinto número de batches.

### Por qué guardar Loss y Dice, no solo uno

La función de pérdida (`DiceBCELoss`) es lo que **optimiza** el modelo, pero no es necesariamente la métrica que os interesa como resultado clínico o de competición. Guardar ambas os permite detectar situaciones donde divergen:

| Situación | Loss | Dice | Interpretación |
|-----------|------|------|-----------------|
| Caso normal | Baja | Sube | El modelo mejora en ambos sentidos |
| Meseta engañosa | Baja poco a poco | Se estanca | La pérdida sigue optimizándose pero no se traduce en mejor segmentación real |
| Divergencia train/val | Train baja, Val sube | Train sube, Val baja | Señal de sobreajuste (lo veréis en detalle en la Práctica 8) |

> **Nota:** Kaggle no evalúa vuestra Loss, evalúa el Dice Score sobre el conjunto de test. La Loss es solo la señal que usáis internamente para entrenar.

### Quedaros con el mejor modelo, no con el último (checkpointing)

El bucle anterior entrena las `NUM_EPOCHS` épocas completas y, al terminar, `model` contiene los pesos de la **última** época — no necesariamente la mejor. Si el Val Dice sube y baja de una época a otra (algo normal, no es un bug), la última época puede ser peor que una anterior ya vista.

La solución es guardar una copia de los pesos cada vez que el Val Dice mejora, y al final del bucle recuperar esa copia:

```python
if epoch_val_dice > best_val_dice:
    best_val_dice = epoch_val_dice
    torch.save(model.state_dict(), 'best_model.pth')
```

`model.state_dict()` es un diccionario con todos los pesos del modelo en ese momento; `torch.save` lo guarda en disco. Al terminar el bucle, `model.load_state_dict(torch.load('best_model.pth'))` sobrescribe los pesos actuales de `model` con los que guardasteis — así, cualquier código posterior (Tarea 7.4, Práctica 9, Práctica 10) que use `model` está usando automáticamente el mejor checkpoint, sin que haga falta cambiar nada más.

> **Idea clave:** guardar solo el número (`max(val_dices)`) no sirve de nada si no guardáis también los pesos que lo produjeron — el número sin los pesos es solo una curiosidad estadística.

### Por qué Google Drive y no el disco de Colab

Estas prácticas se hacen en días distintos. Cada día que abrís Colab es un **runtime nuevo**: el disco local (`/content/...`) se borra por completo, así que un checkpoint guardado ahí solo sobrevive dentro de la sesión en la que lo creasteis. Si guardarais el checkpoint solo en local, tendríais que reentrenar el modelo desde cero cada día únicamente para volver a tener `model` listo, antes incluso de poder empezar la tarea nueva del día.

Google Drive sí persiste entre sesiones. La carpeta `DRIVE_FOLDER` ya la montasteis en la Práctica 6 para descargar el dataset — reutilizamos esa misma carpeta para guardar también los checkpoints, en una subcarpeta `checkpoints/`.

Con esto, cada celda de entrenamiento sigue esta lógica:

| Situación | Qué hace la celda |
|-----------|---------------------|
| No existe checkpoint en Drive (primera vez) | Entrena normalmente las `NUM_EPOCHS` épocas y, al terminar, guarda pesos + historial (Loss/Dice) en Drive |
| Ya existe checkpoint en Drive (sesión de un día anterior) | Carga directamente los pesos y el historial guardado — **no vuelve a entrenar** |

> **Idea clave:** el checkpoint en Drive no es solo los pesos del modelo — también incluye el historial de `train_losses`, `val_losses`, `train_dices` y `val_dices`, porque las gráficas y comparativas de tareas posteriores (por ejemplo, la Tarea 8.4) necesitan esas listas aunque el modelo se haya entrenado días atrás.

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| `zero_grad → forward → loss → backward → step` | Orden fijo; saltarse `zero_grad` acumula gradientes entre batches |
| Época vs batch vs paso | Una época = `len(train_loader)` pasos, no el número de imágenes |
| `model.train()` / `model.eval()` | Cambia el comportamiento de BatchNorm; obligatorio alternarlos entre fases |
| `torch.no_grad()` | Evita construir el grafo de gradientes en validación (memoria y velocidad) |
| Acumular y dividir | Sumar por batch, dividir por `len(loader)` al final de la época |
| Loss vs Dice | La Loss optimiza, el Dice es lo que realmente se evalúa |
| Checkpointing | Guardar los pesos del mejor Val Dice y recargarlos al final; si no, os quedáis con los de la última época |
| Checkpoint en Drive | Persiste entre sesiones de días distintos; la celda comprueba si ya existe y evita reentrenar si es así |


## Tarea 7.3: Bucle de Entrenamiento
Es hora de entrenar.
1. Crea un bucle para `NUM_EPOCHS` (empieza con 5 o 10 para la clase, es para poder comprobar que funciona bien).
2. En cada época, itera sobre el `train_loader`: forward pass, calcular loss, backward pass y `optimizer.step()`.
3. Al final de cada época, evalúa el modelo en el `val_loader` (en modo `model.eval()` y `torch.no_grad()`).
4. Guarda la pérdida de train y val en listas para graficar al final. Es interesante observar el dice tanto en el entrenamiento como en la validación ya que es lo que kaggle va a valorar.
5. Guarda los pesos del modelo (`torch.save(model.state_dict(), ...)`) cada vez que el Val Dice de validación mejore respecto a las épocas anteriores, y al terminar el bucle carga esos pesos en `model` con `load_state_dict`. Así os quedáis con el mejor modelo, no con el de la última época.
6. Guarda el checkpoint final (pesos + historial) en la carpeta de Google Drive donde está el dataset (`DRIVE_FOLDER`, montada en la P6), no solo en el disco local de Colab — así persiste entre sesiones de días distintos. Al principio de la celda, comprueba si ya existe un checkpoint en Drive de una sesión anterior: si existe, cárgalo directamente (junto con el historial de Loss/Dice) y no entrenes de nuevo.

> **CHECKPOINT P7.1:** Mostrar al profesor las gráficas de pérdida (Loss) decreciendo (entrenamiento y validación) y el Dice Score creciendo (entrenamiento y validación)

In [ ]:
# ============================================================
# TAREA 7.3: ENTRENAMIENTO
# ============================================================
from pathlib import Path

NUM_EPOCHS = 10 # Empezamos con pocas épocas para ver resultados rápido

# Carpeta de checkpoints en Google Drive (persiste entre sesiones/días distintos)
CHECKPOINT_DIR = Path(DRIVE_FOLDER) / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_PATH = CHECKPOINT_DIR / 'baseline_checkpoint.pth'

train_losses = []
val_losses = []
train_dices = []
val_dices = []

if DRIVE_CHECKPOINT_PATH.exists():
    # Ya se entrenó este modelo en una sesión anterior: cargarlo en vez de reentrenar
    print(f"Checkpoint encontrado en Google Drive: {DRIVE_CHECKPOINT_PATH}")
    # Tu código aquí: torch.load(DRIVE_CHECKPOINT_PATH) y restaurar model, train_losses,
    # val_losses, train_dices, val_dices y best_val_dice desde el checkpoint

else:
    # 1. Variables para quedarte con el mejor modelo (no el de la última época)
    # Tu código aquí: best_val_dice = -1 y BEST_MODEL_PATH = 'best_model_baseline.pth'

    print("Iniciando entrenamiento...")
    # 2. Crear lazo de entrenamiento
    for epoch in range(NUM_EPOCHS):
        # 3. Fase de Entrenamiento (model.train())
        # Tu código aquí para iterar train_loader y actualizar pesos

        # 4. Fase de Validación (model.eval() y torch.no_grad())
        # Tu código aquí para iterar val_loader y calcular loss sin gradientes

        # 5. guardar en listas para graficar despues
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        # 6. Guardar checkpoint local si el Val Dice de esta época es el mejor hasta ahora
        # Tu código aquí: comparar epoch_val_dice con best_val_dice y, si mejora,
        # actualizar best_val_dice y hacer torch.save(model.state_dict(), BEST_MODEL_PATH)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")

    # 7. Cargar los pesos del mejor modelo local (no os quedéis con los de la última época)
    # Tu código aquí: model.load_state_dict(torch.load(BEST_MODEL_PATH))

    # 8. Guardar el checkpoint final (pesos + historial) en Google Drive para no reentrenar en próximas sesiones
    # Tu código aquí: torch.save({...}, DRIVE_CHECKPOINT_PATH)

# Graficar resultados
import matplotlib.pyplot as plt
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

## Bloque 7.4: De Entrenamiento a Submission
### Qué cambia (y qué no) al pasar a inferencia

Entrenar y predecir sobre test comparten casi todo el código, con tres diferencias clave:

| | Entrenamiento / Validación | Inferencia sobre test |
|---|---|---|
| Modo del modelo | `model.train()` en train, `model.eval()` en val | `model.eval()` siempre |
| Gradientes | Se calculan en train (`loss.backward()`) | Nunca (`torch.no_grad()`) |
| ¿Hay `masks` reales? | Sí, para calcular loss y Dice | No — el test no trae máscara |
| Salida que interesa | Un número (loss, Dice) para comparar modelos | Una máscara binaria para enviar a Kaggle |

> **Nota:** al no tener máscara real en test, no podéis calcular vuestro Dice Score localmente sobre estas imágenes. La única forma de saber cómo de bien lo hace vuestro modelo ahí es subir la submission y mirar el Leaderboard.

### El preprocesamiento debe ser idéntico al de entrenamiento

Este es el punto donde más fácil es introducir un bug silencioso. El modelo aprendió a partir de imágenes preprocesadas de una manera concreta (Bloque 7.1: resize a `IMG_SIZE`, normalización con media/std de ImageNet). Si en inferencia preprocesáis distinto — aunque sea "solo" saltaros la normalización ImageNet y quedaros en `/255.0` — el modelo recibe datos con una distribución distinta a la que vio en entrenamiento, y su rendimiento cae sin que salte ningún error.

```python
# Debe ser el MISMO preprocesamiento que en BUSIDataset (Bloque 7.1), paso a paso:
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_norm = img_resized.astype(np.float32) / 255.0
img_norm = (img_norm - mean) / std          # las mismas mean/std de ImageNet
img_tensor = torch.tensor(img_norm).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
```

> **Pregunta para pensar:** si entrenáis con normalización ImageNet pero en inferencia solo dividís por 255, ¿el modelo dará un error, o simplemente predicciones peores sin avisar? ¿Por qué es más peligroso el segundo caso?

### La máscara predicha no tiene el tamaño original

El modelo siempre trabaja a `IMG_SIZE x IMG_SIZE` (por ejemplo 256x256), pero cada imagen de test tiene su propio tamaño original. Kaggle espera la máscara en la resolución original de la imagen, así que hay que deshacer el resize antes de codificar en RLE:

```
Imagen original (H0, W0)
        │  resize a IMG_SIZE
        ▼
Modelo → máscara (IMG_SIZE, IMG_SIZE)
        │  resize de vuelta a (H0, W0)
        ▼
Máscara en tamaño original → mask_to_rle()
```

Guardad `original_h, original_w` **antes** de redimensionar la imagen de entrada — es fácil olvidarlo y quedarse solo con el tamaño ya reescalado.

### De probabilidad a máscara binaria

El modelo no devuelve directamente una máscara — devuelve **logits**, que hay que convertir en dos pasos:

| Paso | Operación | Resultado |
|------|-----------|-----------|
| 1. Logits → probabilidad | `torch.sigmoid(logits)` | Valores continuos entre 0 y 1 |
| 2. Probabilidad → máscara binaria | `(probs > threshold)` | Valores 0 o 1 |

Aquí usamos `threshold = 0.5` porque es el valor por defecto razonable, pero no hay ninguna garantía de que sea el mejor corte para este problema — en la Práctica 9 buscaréis el `best_threshold` específico para vuestro modelo.

### Reutilizar `mask_to_rle`: por qué no reescribirla

La función `mask_to_rle` ya la implementasteis en la Práctica 6. La idea de un `Dataset`/pipeline bien organizado es precisamente esta: cada pieza (carga de datos, modelo, codificación RLE) se escribe una vez y se reutiliza en todas las prácticas siguientes. Si tenéis que copiar y pegar la misma lógica de RLE en cada práctica, es una señal de que conviene guardarla en una celda de utilidades al principio del notebook en vez de repetirla.

### El formato de `submission.csv`

Kaggle valida la submission comparando columnas exactas contra `sample_submission.csv`:

| Columna | Contenido |
|---------|-----------|
| `Id` | Nombre del fichero de imagen (debe coincidir exactamente con el de test) |
| `Expected` | El string RLE de la máscara predicha para esa imagen |

Una fila con un `Id` mal escrito, o un RLE con índices que no encajan con las dimensiones declaradas, hace que Kaggle rechace o puntúe mal esa fila — merece la pena comparar vuestro `submission.csv` contra `sample_submission.csv` (mismo número de filas, mismos `Id`) antes de subirlo.

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Test set | No tiene máscaras; solo sabréis vuestro Dice real al subir a Kaggle |
| Preprocesamiento | Debe ser idéntico al de entrenamiento (resize + normalización), o el modelo rinde peor sin dar error |
| Redimensionar la máscara | El modelo predice a `IMG_SIZE`; hay que devolver la máscara al tamaño original antes del RLE |
| Logits → máscara | `sigmoid()` da probabilidades, un `threshold` las convierte en máscara binaria |
| `mask_to_rle` | Reutilizar la de la Práctica 6, no reescribirla |
| `submission.csv` | Columnas `Id` y `Expected`; comparar contra `sample_submission.csv` antes de subir |




## Tarea 7.4: Inferencia y Submission a Kaggle
¡El modelo está entrenado! Ahora hay que predecir sobre las imágenes de test.
1. Itera sobre las imágenes de la carpeta `test/images`.
2. Pasa la imagen por el modelo (recuerda preprocesarla igual que en train).
3. Aplica `torch.sigmoid()` a la salida y umbraliza (ej. > 0.5) para obtener la máscara binaria.
4. Redimensiona la máscara al tamaño original de la imagen.
5. Convierte la máscara a formato RLE usando la función `mask_to_rle` de la Práctica 6.
6. Guarda los resultados en un DataFrame y expórtalo como `submission.csv`.

> **CHECKPOINT P7.2:** Descarga el `submission.csv` y súbelo a Kaggle. ¡Muestra tu posición en el Leaderboard!

In [ ]:
# ============================================================
# TAREA 7.4: INFERENCIA Y SUBMISSION
# ============================================================
import os
import pandas as pd

# Asegúrate de tener la función mask_to_rle definida (cópiala de la P6 si es necesario)
# def mask_to_rle(mask): ...

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = list(test_img_dir.glob('*.png'))

results = []

print("Generando predicciones para el test set...")
model.eval()
with torch.no_grad():
    for img_path in test_images:
        # 1. Cargar y preprocesar imagen (idéntico al preprocesamiento de BUSIDataset, Bloque 7.1)
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_norm = img_resized.astype(np.float32) / 255.0

        # Normalización ImageNet: debe ser IDÉNTICA a la de BUSIDataset (Bloque 7.1),
        # o el modelo recibe datos con una distribución distinta a la de entrenamiento
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img_norm - mean) / std

        img_tensor = torch.tensor(img_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        # 2. Predecir
        # Tu código aquí (logits -> sigmoid -> threshold > 0.5)

        # 3. Redimensionar máscara al tamaño original
        # Tu código aquí

        # 4. Convertir a RLE y guardar
        rle = mask_to_rle(mask_final)
        results.append({'Id': img_path.name, 'Expected': rle})

# Crear DataFrame y guardar
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)
print(" submission.csv generado correctamente.")